In [0]:
storage_account = "dlforformula1"
container = "bronze"
client_id = "16e41178-4311-45a7-b2ec-ff3999a7593b"
tenant_id = "0bf495b2-f9e4-4786-82ea-d1ff2c6f9353"
account_fqdn = f"{storage_account}.dfs.core.windows.net"

# (Optional but helpful) remove any key-based config that may be set on cluster
for k in [
    f"fs.azure.account.key.{account_fqdn}",
    f"fs.azure.account.keyprovider.{account_fqdn}",
    "fs.azure.account.key"
]:
    try:
        spark.conf.unset(k)
    except:
        pass

# Set OAuth configs (IMPORTANT: include the account FQDN in the key)
spark.conf.set(f"fs.azure.account.auth.type.{account_fqdn}", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{account_fqdn}",
               "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{account_fqdn}", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{account_fqdn}",
               dbutils.secrets.get(scope="adls-scope", key="client-secret"))
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{account_fqdn}",
               f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

###### Read the races csv file

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType

In [0]:
races_schema = StructType([
    StructField("raceId", IntegerType(), True),
    StructField("year", IntegerType(), True),
    StructField("round", IntegerType(), True),
    StructField("circuitId", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("date", DateType(), True),
    StructField("time", StringType(), True),
    StructField("url", StringType(), True)
])

In [0]:
# df = (spark.read.schema(races_schema).format("csv").options(
#             header=True,
#             delimiter=",").load("abfss://demo@dlforformula1.dfs.core.windows.net/races.csv"))
races_df = (spark.read.schema(races_schema).format("csv").options(
            header=True,
            delimiter=",").load("abfss://bronze@dlforformula1.dfs.core.windows.net/races.csv"))
# display(df)
print(races_df.columns)
print(races_df.dtypes)
races_df.printSchema()

In [0]:
from pyspark.sql.functions import current_timestamp, to_timestamp, concat, col, lit

races_df = races_df.withColumn("ingested_date", current_timestamp())\
    .withColumn("race_timestamp", to_timestamp(concat(col("date"), lit(" "), col("time")), "yyyy-MM-dd HH:mm:ss"))
# races_df.show(truncate = False)
races_df.columns



In [0]:
races_selected_df = races_df.select(
    col('raceId').alias('race_id'),
    col('circuitId').alias('circuit_id'),
    col('year').alias(race_year),
    col('round'),
    col('name'),
    col('ingested_date'),
    col('race_timestamp')
)
races_selected_df.show()

In [0]:
# write to silver layer
races_selected_df.write.mode('overwrite').format('parquet').save("abfss://silver@dlforformula1.dfs.core.windows.net/races")



##### How to use partionBy in spark?

In [0]:
races_selected_df.write.mode('overwrite').partitionBy('race_year').save("abfss://silver@dlforformula1.dfs.core.windows.net/races_partionby")
